In [1]:
import pandas as pd
import json
import glob


In [14]:
file_paths = glob.glob('../data/processed_data/*')

In [15]:
sdoh_category = set()

In [16]:
sdoh_label_dict = {}
for path in file_paths:  
    f = json.load(open(path, 'r'))
    for idx, sinfo in f.items():
        # check if SDoH category exist in this sentence:
        if 'SDoH' in sinfo.keys():
            sdohs = sinfo['SDoH']
            for sdoh in sdohs:
                for category, detail in sdoh.items():
                    sdoh_category.add(category)
                    if category not in sdoh_label_dict.keys():
                        sdoh_label_dict[category] = {}
                    for label, value in detail.items():
                        if label not in sdoh_label_dict[category].keys():
                            sdoh_label_dict[category][label] = set()
                        sdoh_label_dict[category][label].add(value)


    

In [17]:
sdoh_label_dict.keys()

dict_keys(['Healthcare', 'Living', 'Smoke', 'Employment', 'Social', 'Education', 'Transportation', 'Mental Health', 'Insurance', 'Financial', 'Substance Use', 'Trauma', 'Adherence', 'Literacy', 'Recommendation', 'Concern'])

In [18]:
category_schemas = {}

In [19]:
for category, properties in sdoh_label_dict.items():
    category_schemas[category] = {
        "name" : f"extract_{category.lower()}_info",
        "description": f"Extract structured {category.lower()} information from a sentence.",
        "parameters": {
                "type": "object",
                "properties": {
                    "category": {"type": "string", "enum":[category]},
                    }
                }
            }
    for k,v in properties.items():
        category_schemas[category]["parameters"]["properties"][k] = {"type": "string", "enum": list(v)}
    

In [20]:
with open("../config/llm_category_schema.json", "w") as f:
    json.dump(category_schemas, f, indent=4)

In [21]:
import json, copy, os, textwrap, re, pathlib

in_path = "../config/llm_category_schema.json"
with open(in_path, "r", encoding="utf-8") as f:
    category_schemas = json.load(f)

In [24]:

# Assuming category_schemas is already generated from sdoh_label_dict
new_schema = copy.deepcopy(category_schemas)

def set_category(cat_key, name, description, props):
    # Updated to use the array wrapper format for multiple extractions
    new_schema[cat_key] = {
        "name": name,
        "description": description + " If there are multiple conditions or events, extract each as a separate item in the list.",
        "parameters": {
            "type": "object",
            "properties": {
                "extracted_conditions": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": props
                    }
                }
            }
        }
    }

# Helper: experiencer enums from existing, if present
def get_experiencer_enum(cat_key, default=None):
    try:
        # Changed 'schema' to 'category_schemas' to avoid NameError
        props = category_schemas[cat_key]["parameters"]["properties"]
        if "Experiencer" in props and "enum" in props["Experiencer"]:
            return props["Experiencer"]["enum"]
    except Exception:
        pass
    return default or ["patients"]

# Financial
set_category(
    "Financial",
    "extract_financial_info",
    "Extract structured financial information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Financial"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Financial")},
        "financial_status": {"type": "string", "enum": ["poverty", "constrain", "normal"]},
    }
)

# Employment
set_category(
    "Employment",
    "extract_employment_info",
    "Extract structured employment information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Employment"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Employment")},
        "employment_status": {"type": "string", "enum": ["employed", "unemployed", "retired", "on leave"]},
    }
)

# Education
set_category(
    "Education",
    "extract_education_info",
    "Extract structured education information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Education"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Education")},
        "education_status": {"type": "string", "enum": ["current", "past", "none", "future"]},
        "education_level": {"type": "string", "enum": ["high_level", "occupational", "general", "childhood"]},
    }
)

# Healthcare
set_category(
    "Healthcare",
    "extract_healthcare_info",
    "Extract structured healthcare information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Healthcare"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Healthcare")},
        "healthcare_type": {"type": "string", "enum": ["medications", "surgeries/procedures", "clinical visits", "counseling", "hospital stay"]},
    }
)

# MentalHealth (rename from "Mental Health" if present)
mh_ex = get_experiencer_enum("Mental Health", default=get_experiencer_enum("Mental Health", default=["patients","parents/caregiver","relatives"]))
set_category(
    "Mental Health",
    "extract_mentalhealth_info",
    "Extract structured mental health information from a sentence.",
    {
        "category": {"type": "string", "enum": ["MentalHealth"]},
        "mentalhealth_type":{"type":"string", "enum": ['general','OCD','PTSD','bipolar','ADHD','sleep problem','anxiety','neuro condition','medication','depression']},
        "Experiencer": {"type": "string", "enum": mh_ex},
        "mentalHealth_status": {"type": "string", "enum": ["none", "current", "past"]},
    }
)
# Remove old key if exists
# if "Mental Health" in new_schema:
#     del new_schema["Mental Health"]

# Social
set_category(
    "Social",
    "extract_social_info",
    "Extract structured social support levels from a sentence.",
    {
        "category": {"type": "string", "enum": ["Social"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Social")},
        "social_family_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "social_church_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "social_government_level": {"type": "string", "enum": ["low", "medium", "high"]},
    }
)

# Living
set_category(
    "Living",
    "extract_living_info",
    "Extract structured living information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Living"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Living")},
        "num_of_caregivers": {"type": "integer"},
    }
)

# Smoke (just rename key)
set_category(
    "Smoke",
    "extract_smoke_info",
    "Extract structured smoke information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Smoke"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Smoke")},
        "smoke_status": {"type": "string", "enum": ["current", "past", "none"]},
    }
)

# SubstanceUse (rename from "Substance Use")
su_ex = get_experiencer_enum("Substance Use", default=get_experiencer_enum("Substance Use", default=["patients","parents/caregiver","grandparents"]))
set_category(
    "Substance Use",
    "extract_substanceuse_info",
    "Extract structured substance use information from a sentence.",
    {
        "category": {"type": "string", "enum": ["SubstanceUse"]},
        "Experiencer": {"type": "string", "enum": su_ex},
        "substanceuse_status": {"type": "string", "enum": ["current", "past", "none"]},
    }
)
# if "Substance Use" in new_schema:
#     del new_schema["Substance Use"]

# Trauma
set_category(
    "Trauma",
    "extract_trauma_info",
    "Extract structured trauma information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Trauma"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Trauma")},
        "divorce": {"type": "string", "enum": ["current", "past", "none"]},
        "loss": {"type": "string", "enum": ["current", "past", "none"]},
        "physical_abuse": {"type": "string", "enum": ["current", "past", "none"]},
        "psychological_abuse": {"type": "string", "enum": ["current", "past", "none"]},
        "domestic_violence": {"type": "string", "enum": ["current", "past", "none"]},
        "dcf": {"type": "string", "enum": ["current", "past", "none"]},
        "abandonment": {"type": "string", "enum": ["current", "past", "none"]},
    }
)

# Insurance
set_category(
    "Insurance",
    "extract_insurance_info",
    "Extract structured insurance information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Insurance"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Insurance")},
        "insurance_type": {"type": "string", "enum": ["public", "private"]},
    }
)

# Adherence
set_category(
    "Adherence",
    "extract_adherence_info",
    "Extract structured adherence information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Adherence"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Adherence")},
        "adherence_medication": {"type": "string", "enum": ["high", "normal", "low", "none"]},
        "adherence_therapy": {"type": "string", "enum": ["high", "normal", "low", "none"]},
        "adherence_other": {"type": "string", "enum": ["high", "normal", "low", "none"]},
    }
)

# Literacy
set_category(
    "Literacy",
    "extract_literacy_info",
    "Extract structured literacy information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Literacy"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Literacy")},
        "transplant_knowledge": {"type": "string", "enum": ["high", "normal", "low", "none"]},
        "caregiving_knowledge": {"type": "string", "enum": ["high", "normal", "low", "none"]},
        "medication_knowledge": {"type": "string", "enum": ["high", "normal", "low", "none"]},
    }
)

# Recommendation
set_category(
    "Recommendation",
    "extract_recommendation_info",
    "Extract structured recommendations from a sentence.",
    {
        "category": {"type": "string", "enum": ["Recommendation"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Recommendation")},
        "increase_literacy": {"type": "string", "enum": ["yes", "no"]},
        "increase_social_support": {"type": "string", "enum": ["yes", "no"]},
        "increase_financial_support": {"type": "string", "enum": ["yes", "no"]},
    }
)

# Concern
set_category(
    "Concern",
    "extract_concern_info",
    "Extract structured concern information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Concern"]},
        "concern_level": {"type": "string", "enum": ["light", "moderate", "intense"]},
    }
)

# Transportation
set_category(
    "Transportation",
    "extract_transportation_info",
    "Extract structured transportation information from a sentence.",
    {
        "category": {"type": "string", "enum": ["Transportation"]},
        "Experiencer": {"type": "string", "enum": get_experiencer_enum("Transportation")},
        "transportation_vehicle_access": {"type": "string", "enum": ["easy", "hard"]},
        "transportation_cost": {"type": "string", "enum": ["low", "high"]},
        "transportation_distance": {"type": "string", "enum": ["short", "long"]},
        "transportation_license": {"type": "string", "enum": ["yes", "no"]},
        "transportation_violation": {"type": "string", "enum": ["yes", "no"]},
    }
)

out_path = "../config/sdoh_extraction_schema_updated.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(new_schema, f, indent=2, ensure_ascii=False)

print(f"Updated schema saved to {out_path}")

Updated schema saved to ../config/sdoh_extraction_schema_updated.json


### Update schema for strictly enforce the enum output

In [5]:
import json
from copy import deepcopy
from pathlib import Path


input_path = Path("../config/sdoh_extraction_schema_updated.json")
output_path = Path("../config/sdoh_extraction_schema_strict.json")


with input_path.open("r", encoding="utf-8") as f:
    category_schemas = json.load(f)


def make_schema_strict(schema):
    schema = deepcopy(schema)

    if schema.get("type") == "object":
        properties = schema.get("properties", {})

        schema["additionalProperties"] = False
        schema["required"] = list(properties.keys())

        for key, value in properties.items():
            properties[key] = make_schema_strict(value)

        schema["properties"] = properties

    elif schema.get("type") == "array":
        if "items" in schema:
            schema["items"] = make_schema_strict(schema["items"])

    return schema


strict_category_schemas = deepcopy(category_schemas)

for category, func_schema in strict_category_schemas.items():
    func_schema["strict"] = True
    func_schema["parameters"] = make_schema_strict(func_schema["parameters"])


with output_path.open("w", encoding="utf-8") as f:
    json.dump(strict_category_schemas, f, indent=2, ensure_ascii=False)


print(f"Saved strict schema to: {output_path}")

Saved strict schema to: ..\config\sdoh_extraction_schema_strict.json
